# CRO analysis - DTC bedding store (case study)
Real public data where it exists (**PUBLIC DATA**), a simulated store modelled on public information where it does not
(**SIMULATED** - never Tediber's numbers). Every table below is produced by `sql/` (DuckDB) or `src/` and saved in
`data/processed/`. Run order: `extract_ga4.py` -> `simulate.py` -> `run_sql.py` -> `analyse.py` -> this notebook.

In [1]:
import sys, pandas as pd, numpy as np
sys.path.insert(0, '../src')
import stats_tests as st
P = '../data/processed/'
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
pct = lambda s: (100 * s).round(2)

## 1. Real funnel (PUBLIC DATA - Google Merchandise Store GA4 sample, not Tediber)
Open funnel: share of sessions reaching each step. Sessions = `user_pseudo_id` + `ga_session_id` (the id alone is not unique).

In [2]:
g = pd.read_csv(P + 'ga4_funnel_segments.csv')
cols = ['sessions', 'view_item', 'add_to_cart', 'begin_checkout', 'add_shipping_info', 'add_payment_info', 'purchase']
show = g[g.dimension.isin(['All', 'Device', 'Visitor type', 'Landing page'])].set_index(['dimension', 'segment'])
(show[cols[1:]].div(show.sessions, axis=0) * 100).round(2)

view_item  add_to_cart  begin_checkout  add_shipping_info  add_payment_info  purchase
dimension    segment                                                                                               
All          All sessions         21.40         4.22            3.09               3.09              1.89      1.35
Device       desktop              21.47         4.21            3.06               3.06              1.86      1.32
             mobile               21.32         4.25            3.13               3.13              1.94      1.39
             tablet               21.26         4.02            3.00               3.00              1.90      1.30
Landing page Product page         25.38         3.12            1.91               1.91              1.35      0.96
             Home                 21.48         6.00            4.58               4.58              2.51      1.78
             No page_view          1.30         0.17            0.00               0.00              0.00      0.00
             Cart / checkout      25.28         9.66           10.92              10.92              8.76      6.24
             Category page        10.48         1.39            1.36               1.36              0.99      0.78
             Other page            5.07         1.26            0.93               0.93              0.52      0.31
Visitor type New                  19.98         3.39            2.29               2.29              1.01      0.68
             Returning            25.16         6.41            5.20               5.19              4.24      3.11

Returning visitors convert 4.6x better than new ones (3.11% vs 0.68%): for a product bought after several visits,
conversion **per user** over a period says more than conversion per session. Mobile and desktop are close in this sample.

In [3]:
pd.read_csv(P + 'ga4_data_quality.csv').assign(value=lambda d: d.value.round(4))

,check_name,value,unit
0,purchase events,5692.0000,events
1,purchase events with a missing or (not set) tr...,906.0000,events
2,distinct transaction ids,4451.0000,ids
3,transaction ids sent more than once,326.0000,ids
4,extra purchase events from duplicated ids,335.0000,events
5,sessions without a session_start event,4990.0000,sessions
6,share of sessions without session_start,0.0139,share
7,share of sessions with channel not available (...,0.2073,share
8,purchase sessions without begin_checkout in th...,3.0000,sessions
9,view_item_list events (list views barely tracked),71.0000,events


In [4]:
pd.read_csv(P + 'ga4_revenue_dedup.csv')

,revenue_all_events_usd,revenue_dedup_usd,inflation_share
0,330348.0,307640.0,0.0738


**Data quality on a real export**: 326 transaction ids are sent more than once (+7.4% revenue if not de-duplicated),
906 purchase events have no usable id, and a fifth of sessions have no channel. These checks come before any funnel reading.

## 2. Weekly report (SIMULATED)

In [5]:
k = pd.read_csv(P + 'weekly_kpis.csv', parse_dates=['week_start'])
a = k[k.dimension == 'All'].set_index('week_start')
a.loc['2026-03-02':'2026-03-23', ['sessions', 'conversion_rate', 'conversion_per_user', 'aov_eur', 'revenue_per_visit_eur',
                                   'net_revenue_per_visit_eur', 'add_to_cart_rate', 'checkout_abandonment', 'conversion_rate_wow']].round(4)

,sessions,conversion_rate,conversion_per_user,aov_eur,revenue_per_visit_eur,net_revenue_per_visit_eur,add_to_cart_rate,checkout_abandonment,conversion_rate_wow
week_start,,,,,,,,,
2026-03-02,62489.0,0.0119,0.0156,408.7800,4.8670,4.5379,0.0392,0.4901,0.0219
2026-03-09,62670.0,0.0108,0.0141,404.2240,4.3796,4.1664,0.0382,0.5268,-0.0900
2026-03-16,62483.0,0.0107,0.0140,430.7622,4.6190,4.3847,0.0380,0.5204,-0.0103
2026-03-23,61509.0,0.0119,0.0156,409.3200,4.8845,4.5527,0.0385,0.4824,0.1129


The topline barely moves in the week of 16 March (conversion -1% week on week). The problem is hidden in one segment:

In [6]:
f = pd.read_csv(P + 'funnel_segments.csv')
f[f.dimension.isin(['Device x browser', 'Category'])].set_index(['dimension', 'segment'])[
    ['sessions', 'pass_add_to_cart', 'pass_checkout', 'pass_shipping', 'pass_payment', 'conversion_rate']].round(4)

sessions  pass_add_to_cart  pass_checkout  pass_shipping  pass_payment  conversion_rate
dimension        segment                                                                                                      
Category         Mattress             2063158.0            0.0562         0.6267         0.7997        0.8153           0.0088
                 Pillow & duvet       1194798.0            0.1174         0.6254         0.8038        0.8207           0.0187
                 Bed base & bed        869676.0            0.0694         0.6251         0.6000        0.8196           0.0083
                 Bed linen             650489.0            0.1259         0.6269         0.8029        0.8178           0.0200
                 Kids                  380369.0            0.0954         0.6282         0.8024        0.8219           0.0151
                 Sofa                  271606.0            0.0479         0.6168         0.5940        0.8254           0.0056
Device x browser mobile / Safari iOS  1694188.0            0.0727         0.5826         0.7679        0.8176           0.0102
                 mobile / Chrome      1367794.0            0.0728         0.5798         0.7692        0.8135           0.0102
                 desktop / Chrome     1256156.0            0.1011         0.6829         0.7691        0.8220           0.0168
                 desktop / Other       590524.0            0.0965         0.6846         0.7705        0.8255           0.0162
                 tablet / Safari iOS   195575.0            0.0863         0.6314         0.7659        0.8062           0.0130
                 mobile / Other        195284.0            0.0695         0.5757         0.7680        0.8205           0.0096
                 tablet / Chrome        97924.0            0.0829         0.6260         0.7781        0.8183           0.0129
                 tablet / Other         32651.0            0.0781         0.6047         0.7668        0.7996           0.0111

Over the whole period, **bulky items (bed bases, beds, sofas) lose twice as many shoppers at the delivery step**
(~40% vs ~20%): a structural leak, not an incident - it goes to the backlog (test C).

## 3. Anomaly detection and how it scored (SIMULATED)
Method (`sql/03_anomalies.sql`): 7-day trailing rate per slice and KPI; expected = median of the same weekday over 8 weeks,
promotion windows excluded (inside a promotion: last year's promotion, scaled); robust z with a MAD / binomial floor;
alert = |z| >= 3.5, change >= 10%, for 5 days or more. Then a traffic-mix check and the release log.

In [7]:
inc = pd.read_csv(P + 'anomaly_incidents.csv')
inc[['slice', 'kpi', 'first_day', 'last_day', 'days_flagged', 'rel_change_at_peak', 'z_at_peak', 'status']].round(3)

,slice,kpi,first_day,last_day,days_flagged,rel_change_at_peak,z_at_peak,status
0,mobile / Safari iOS,payment step,2026-03-12,2026-03-21,10,-0.227,-8.126,open: investigate
1,mobile / Safari iOS,conversion rate,2026-03-15,2026-03-20,6,-0.360,-4.992,open: investigate
2,mobile / Mattress,add-to-cart rate,2026-05-17,2026-05-25,9,-0.228,-4.192,open: investigate
3,Bed linen,add-to-cart rate,2026-09-02,2026-09-06,5,-0.191,-4.974,"explained: traffic mix, no incident opened"
4,mobile,add-to-cart rate,2026-09-02,2026-09-16,15,-0.158,-6.759,"explained: traffic mix, no incident opened"
5,mobile / Chrome,add-to-cart rate,2026-09-02,2026-09-08,7,-0.181,-4.797,"explained: traffic mix, no incident opened"
6,New,add-to-cart rate,2026-09-02,2026-09-16,15,-0.184,-8.165,"explained: traffic mix, no incident opened"
7,All traffic,add-to-cart rate,2026-09-03,2026-09-17,15,-0.168,-9.919,"explained: traffic mix, no incident opened"
8,desktop,add-to-cart rate,2026-09-03,2026-09-14,12,-0.189,-7.243,"explained: traffic mix, no incident opened"
9,Pillow & duvet,add-to-cart rate,2026-09-04,2026-09-09,6,-0.171,-5.672,"explained: traffic mix, no incident opened"


In [8]:
pd.read_csv(P + 'mix_decomposition.csv').round(5)

,first_day,last_day,channel,share_before,share_during,cr_before,cr_during,rate_effect,mix_effect,topline_cr_before,topline_cr_during
0,2026-09-01,2026-09-15,Affiliation,0.08843,0.07091,0.01425,0.01345,-0.00007,-0.00024,0.01181,0.01010
1,2026-09-01,2026-09-15,Direct,0.16950,0.13309,0.01636,0.01652,0.00003,-0.00060,0.01181,0.01010
2,2026-09-01,2026-09-15,Email / CRM,0.10163,0.07820,0.01805,0.02021,0.00022,-0.00047,0.01181,0.01010
3,2026-09-01,2026-09-15,Organic search,0.28048,0.21779,0.01020,0.00949,-0.00020,-0.00059,0.01181,0.01010
4,2026-09-01,2026-09-15,Paid search,0.22035,0.17198,0.01112,0.01093,-0.00004,-0.00053,0.01181,0.01010
5,2026-09-01,2026-09-15,Paid social,0.13963,0.32803,0.00451,0.00433,-0.00003,0.00082,0.01181,0.01010
6,2026-09-19,2026-10-10,Affiliation,0.07461,0.08987,0.01452,0.01305,-0.00011,0.00020,0.01057,0.01156
7,2026-09-19,2026-10-10,Direct,0.14294,0.16607,0.01628,0.01633,0.00001,0.00038,0.01057,0.01156
8,2026-09-19,2026-10-10,Email / CRM,0.08419,0.09840,0.01873,0.01859,-0.00001,0.00026,0.01057,0.01156
9,2026-09-19,2026-10-10,Organic search,0.23376,0.27600,0.00985,0.00985,0.00000,0.00042,0.01057,0.01156


Early September: topline conversion falls, yet inside each channel it is flat (rate effects ~0); the fall is the
**mix effect** of four times more paid-social sessions. Correct handling: explain it, open no incident.

In [9]:
pd.read_csv(P + 'reconciliation_incidents.csv')

,first_day,last_day,days_flagged,max_revenue_ratio,max_duplicate_share,candidate_releases
0,2026-07-08,2026-08-09,33,1.097941,0.117209,2026-07-07 Tracking: Purchase tag migrated to ...


In [10]:
sc = pd.read_csv(P + 'detection_scorecard.csv')
sc[['event_id', 'event', 'expected_handling', 'handled_correctly', 'days_to_detect', 'how']]

,event_id,event,expected_handling,handled_correctly,days_to_detect,how
0,1,Mobile payment broken (Safari iOS),raise,True,2.0,"incident on mobile / Safari iOS, payment step"
1,2,Slower mattress product page,raise,True,5.0,"incident on mobile / Mattress, add-to-cart rat..."
2,3,Delivery fees revealed at checkout (bulky items),funnel finding,True,NaN,funnel by category: shipping-step abandonment ...
3,4,False signal: traffic mix,do not raise,True,NaN,"9 alerts, all triaged as traffic mix; no incid..."
4,5,Duplicate purchase events,raise,True,1.0,reconciliation: analytics revenue above the sh...
5,6,Sample ratio mismatch,invalidate test,True,NaN,"SRM check: variant share 48.2%, p = 5.5e-40"
6,7,The winner that loses,iterate,True,NaN,conversion +8.1% (p = 0.004) but net RPV +2.9%...
7,8,Normal seasonality (Black Month),do not raise,True,NaN,0 incidents opened during Black Month 2026 (ba...
8,99,Other incidents opened (false positives),none,True,NaN,0 other open incidents over 18 months


All 8 planted events are handled as they should be (the answer key was hidden during the analysis and used only here):
the two real breaks are found 2 and 5 days after they start and linked to the right release, the duplicate purchases the day
after, the mix effect and Black Month raise no incident, and no other incident is opened in 18 months.

## 4. Testing it right (SIMULATED)
### Sample size and duration, decided before launch

In [11]:
pd.read_csv(P + 'power_plan.csv')

,test_id,primary_kpi,baseline,mde_relative,alpha,power,visitors_per_arm_needed,eligible_visitors_per_day,planned_days,days_run,visitors_observed
0,A,add-to-cart rate,0.090,0.07,0.05,0.8,33406,3100,28,28,86580
1,B,conversion rate,0.020,0.12,0.05,0.8,56548,4600,28,28,129187
2,C,conversion rate,0.016,0.16,0.05,0.8,40672,1800,49,49,87487
3,D,conversion rate,0.016,0.08,0.05,0.8,156774,6000,56,56,336107


In [12]:
# the same calculation by hand for test D: 1.6% baseline, +8% relative MDE
st.sample_size_per_arm(0.016, 0.08), st.mde_for_sample(0.016, 168_000)

(156774, np.float64(0.07580568075069975))

### Peeking: an A/A test checked every day

In [13]:
pk = st.peeking_false_positive_rate()
print(f"read once at the end: {pk['fixed_horizon']:.1%} false positives; stop at the first p < 0.05: {pk['daily_peeking']:.1%}")

read once at the end: 5.0% false positives; stop at the first p < 0.05: 26.9%


### Read-outs: SRM first, then the primary KPI, then guardrails

In [14]:
ro = pd.read_csv(P + 'experiment_readout.csv')
ro[['test_id', 'variant_share', 'srm_p_value', 'primary_kpi', 'primary_lift', 'primary_ci_low', 'primary_ci_high', 'primary_p_value',
    'rpv_net_lift', 'rpv_net_ci_low', 'rpv_net_ci_high', 'return_rate_control', 'return_rate_variant', 'decision']].round(4)

,test_id,variant_share,srm_p_value,primary_kpi,primary_lift,primary_ci_low,primary_ci_high,primary_p_value,rpv_net_lift,rpv_net_ci_low,rpv_net_ci_high,return_rate_control,return_rate_variant,decision
0,A,0.5002,0.9188,add-to-cart rate,0.0833,0.0402,0.1264,0.0002,0.0350,-0.0782,0.1650,0.0596,0.0759,ship
1,B,0.4816,0.0000,conversion rate,0.0310,-0.0457,0.1077,0.4278,0.0195,-0.0668,0.1161,0.0667,0.0905,invalid: fix and relaunch
2,C,0.4987,0.4308,conversion rate,0.1979,0.0876,0.3082,0.0004,0.2143,0.0784,0.3774,0.0863,0.0748,ship
3,D,0.4994,0.4913,conversion rate,0.0809,0.0262,0.1356,0.0038,0.0293,-0.0343,0.0965,0.0764,0.1086,iterate


In [15]:
# recompute test D by hand from the counts (z test) to check the table
s = pd.read_csv(P + 'experiment_summary.csv').set_index(['test_id', 'variant'])
c, v = s.loc[('D', 'control')], s.loc[('D', 'variant')]
st.two_proportion_test(int(c.orders), int(c.visitors), int(v.orders), int(v.visitors))

{'rate_a': 0.01563716545915972,
 'rate_b': 0.016901593051103937,
 'diff': 0.0012644275919442179,
 'lift': 0.08086040882835066,
 'z': 2.897268245587798,
 'p_value': np.float64(0.003764277434264727),
 'ci_diff_low': np.float64(0.0004090307767837013),
 'ci_diff_high': np.float64(0.0021198244071047347),
 'ci_lift_low': np.float64(0.026157603681561416),
 'ci_lift_high': np.float64(0.13556321397513985)}

**Test D, "the winner that loses"**: conversion +8% (p = 0.004) but 100-night returns go from 7.6% to 10.9%, so net
revenue per visitor is +2.9% with an interval from -3.4% to +9.6%: not shown to be better. Decision: iterate (add firmness
guidance, re-test on net revenue). **Test B** is not read at all: 48/52 split, chi-square p < 0.001 (a redirect lost visitors).

## 5. Business impact, as a range

In [16]:
pd.read_csv(P + 'business_impact.csv').round(0)

,test_id,test_name,eligible_visitors_per_year,net_rpv_control_eur,impact_low_eur,impact_central_eur,impact_high_eur,note
0,A,Sticky add-to-cart button on mobile product pages,1131500,10.0,-910748.0,406927.0,1921376.0,SIMULATED store; range = 95% bootstrap interva...
1,C,Delivery fee shown on bulky-item product pages,657000,9.0,476653.0,1302737.0,2294090.0,SIMULATED store; range = 95% bootstrap interva...


Annualised net revenue of the shipped tests on the simulated store, from the 95% bootstrap interval. Test A's range
includes zero on revenue (its primary metric, add-to-cart, is clearly up): we commit to the range, not to the central value,
and follow conversion after launch.